# PINN Training for the NLSE

**Project**: PINN-NLSE  
**Purpose**: PINN implementation, training, and data-augmented recovery

This notebook documents the Physics-Informed Neural Network trained to solve the
normalized Nonlinear Schrodinger Equation (NLSE):

$$ i\,\frac{\partial u}{\partial \xi} + \frac{s}{2}\,\frac{\partial^2 u}{\partial \tau^2} + N^2\,|u|^2\,u \;=\; 0 $$

**Architecture**: 5 hidden layers x 128 neurons with `tanh`. Inputs `(xi, tau)`, outputs `(a, b)` where `u = a + ib`. **66,690** trainable parameters.

**Loss**:
$$ \mathcal{L} = \lambda_{\text{phys}}\overline{r_a^2 + r_b^2} + \lambda_{\text{ic}}\overline{(a - a_0)^2 + (b - b_0)^2} + \lambda_{\text{bc}}\overline{a^2 + b^2}\Big|_{\tau=\pm \tau_{\max}} + \lambda_{\text{data}}\overline{(a - a_{\text{SSFM}})^2 + \dots} $$

with residuals

$$ r_a = -\partial_\xi b + (s/2)\partial^2_\tau a + N^2(a^2+b^2)\,a, \qquad r_b = +\partial_\xi a + (s/2)\partial^2_\tau b + N^2(a^2+b^2)\,b. $$

**Training strategy**: Adam (coarse, fast) -> L-BFGS (fine, curvature-aware), with mandatory smoke preflight before each baseline run.

**Reproducibility**: this notebook loads pretrained weights from `models/`. If the weights are missing, it runs the smoke profile to produce demo-quality weights (clearly labeled). Final published weights come from `python -m src.train --case <case> --profile baseline` run from the command line.


In [1]:
import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import torch

from src.config import (
    FIGURE_PATHS, N_SOLITON, S_SIGN, TAU_MAX, XI_MAX,
)
from src.pinn_nlse import PINN_NLSE
from src.data_gen import (
    generate_collocation_points, generate_ic_points, generate_bc_points,
    load_ground_truth_npz,
)
from src.utils import compute_relative_l2_error, compute_masked_relative_l2_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


def _resolve_weights(case):
    """Find weights for `case`, preferring frozen 'published' copies.

    Search order (first hit wins):
      1. models/published/<case>_data_augmented_final.pt   (frozen release artifact)
      2. models/published/<case>_final.pt                  (frozen pure PINN)
      3. models/<case>_data_augmented_final.pt             (latest local)
      4. models/<case>_final.pt                            (latest local pure)
    Returns (path, is_data_augmented, is_published).
    """
    candidates = [
        (Path(f'models/published/{case}_data_augmented_final.pt'), True, True),
        (Path(f'models/published/{case}_final.pt'),                  False, True),
        (Path(f'models/{case}_data_augmented_final.pt'),             True, False),
        (Path(f'models/{case}_final.pt'),                            False, False),
    ]
    for p, is_aug, is_pub in candidates:
        if p.exists():
            return p, is_aug, is_pub
    return None, None, None


def _resolve_history(case, augmented, published):
    suffix = 'data_augmented_' if augmented else ''
    base_dir = 'logs/published' if published else 'logs'
    return Path(f'{base_dir}/{case}_{suffix}training_history.json')


soliton_w, soliton_aug, soliton_pub = _resolve_weights('soliton')
gauss_w, gauss_aug, gauss_pub = _resolve_weights('gaussian_dispersion')
missing = []
if soliton_w is None:
    missing.append('soliton')
if gauss_w is None:
    missing.append('gaussian_dispersion')
if missing:
    profile = os.getenv('PINN_NLSE_NOTEBOOK_PROFILE', 'smoke')
    augmented = os.getenv('PINN_NLSE_NOTEBOOK_DATA_AUGMENTED', '0') == '1'
    print(f'Missing weights: {missing}. Training profile = {profile}, data_augmented = {augmented}.')
    from src.train import run_soliton_training, run_gaussian_dispersion_training
    if 'soliton' in missing:
        run_soliton_training(profile=profile, data_augmented=augmented)
    if 'gaussian_dispersion' in missing:
        run_gaussian_dispersion_training(profile=profile, data_augmented=augmented)
    soliton_w, soliton_aug, soliton_pub = _resolve_weights('soliton')
    gauss_w, gauss_aug, gauss_pub = _resolve_weights('gaussian_dispersion')
else:
    print(f'soliton weights : {soliton_w}  (data-augmented={soliton_aug}, published={soliton_pub})')
    print(f'gaussian weights: {gauss_w}  (data-augmented={gauss_aug}, published={gauss_pub})')

model = PINN_NLSE(n_hidden=5, n_neurons=128, s=S_SIGN, N_sq=float(N_SOLITON**2),
                  xi_max=XI_MAX, tau_max=TAU_MAX).to(device)
print(f'Soliton model: {model.count_parameters():,} trainable parameters')


Device: cpu
soliton weights : models\published\soliton_data_augmented_final.pt  (data-augmented=True, published=True)
gaussian weights: models\published\gaussian_dispersion_data_augmented_final.pt  (data-augmented=True, published=True)
Soliton model: 66,690 trainable parameters
